# Export Claim Detector to ONNX

Converts the fine-tuned DeBERTa-v3-base PyTorch checkpoint to ONNX format.

In [1]:
from pathlib import Path

import numpy as np
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_DIR = Path("claim_detector_model")
ONNX_PATH = MODEL_DIR / "model.onnx"
MAX_LENGTH = 256

print(f"Model dir: {MODEL_DIR}")
print(f"ONNX output: {ONNX_PATH}")

Model dir: claim_detector_model
ONNX output: claim_detector_model/model.onnx


## 1. Load PyTorch model

In [2]:
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))
model = AutoModelForSequenceClassification.from_pretrained(str(MODEL_DIR))
model.eval()

print(f"Model type: {model.config.model_type}")
print(f"Labels: {model.config.id2label}")
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Model type: deberta-v2
Labels: {0: 'NOT_CLAIM', 1: 'CLAIM'}
Params: 184,423,682


## 2. Export to ONNX

In [3]:
# Create dummy input for tracing
dummy = tokenizer(
    "NASA confirmed water exists on Mars",
    return_tensors="pt",
    truncation=True,
    max_length=MAX_LENGTH,
)

print(f"Exporting to {ONNX_PATH}...")

torch.onnx.export(
    model,
    (dummy["input_ids"], dummy["attention_mask"]),
    str(ONNX_PATH),
    input_names=["input_ids", "attention_mask"],
    output_names=["logits"],
    dynamic_axes={
        "input_ids": {0: "batch", 1: "seq_len"},
        "attention_mask": {0: "batch", 1: "seq_len"},
        "logits": {0: "batch"},
    },
    opset_version=14,
    do_constant_folding=True,
)

size_mb = ONNX_PATH.stat().st_size / 1024 / 1024
print(f"Exported: {ONNX_PATH} ({size_mb:.0f} MB)")

Exporting to claim_detector_model/model.onnx...


/tmp/ipykernel_37424/2205888726.py:11: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(


Exported: claim_detector_model/model.onnx (704 MB)


## 3. Verify ONNX model

In [4]:
import onnx

onnx_model = onnx.load(str(ONNX_PATH))
onnx.checker.check_model(onnx_model)
print("ONNX model is valid")

print(f"\nInputs:")
for inp in onnx_model.graph.input:
    print(f"  {inp.name}: {[d.dim_param or d.dim_value for d in inp.type.tensor_type.shape.dim]}")
print(f"Outputs:")
for out in onnx_model.graph.output:
    print(f"  {out.name}: {[d.dim_param or d.dim_value for d in out.type.tensor_type.shape.dim]}")

ONNX model is valid

Inputs:
  input_ids: ['batch', 'seq_len']
  attention_mask: ['batch', 'seq_len']
Outputs:
  logits: ['batch', 2]


## 4. Compare PyTorch vs ONNX scores

In [5]:
import onnxruntime as ort

session = ort.InferenceSession(str(ONNX_PATH))

test_texts = [
    "NASA confirmed water exists on Mars",
    "The unemployment rate dropped to 3.4% in January 2023",
    "I think this movie was really great",
    "Good morning everyone!",
    "asdfghjkl random words here",
    "Russia invaded Ukraine on February 24, 2022",
]


def score_pytorch(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)
    return float(probs[0][1])


def score_onnx(text):
    inputs = tokenizer(text, return_tensors="np", truncation=True, max_length=MAX_LENGTH)
    logits = session.run(None, {"input_ids": inputs["input_ids"], "attention_mask": inputs["attention_mask"]})[0]
    exp = np.exp(logits[0] - logits[0].max())  # numerically stable softmax
    probs = exp / exp.sum()
    return float(probs[1])


print(f"{'Text':<55} {'PyTorch':>8} {'ONNX':>8} {'Delta':>8}")
print("-" * 83)
for text in test_texts:
    pt = score_pytorch(text)
    ox = score_onnx(text)
    delta = abs(pt - ox)
    match = "OK" if delta < 0.001 else "DIFF"
    print(f"{text[:53]:<55} {pt:>8.4f} {ox:>8.4f} {delta:>7.5f} {match}")

2026-04-10 18:19:05.433341116 [W:onnxruntime:Default, device_discovery.cc:325 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:92 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


Text                                                     PyTorch     ONNX    Delta
-----------------------------------------------------------------------------------
NASA confirmed water exists on Mars                       1.0000   1.0000 0.00000 OK
The unemployment rate dropped to 3.4% in January 2023     1.0000   1.0000 0.00000 OK
I think this movie was really great                       0.0002   0.0002 0.00000 OK
Good morning everyone!                                    0.0002   0.0002 0.00000 OK
asdfghjkl random words here                               0.0006   0.0006 0.00000 OK
Russia invaded Ukraine on February 24, 2022               0.9999   0.9999 0.00000 OK


## 5. Benchmark latency

In [6]:
import time

N = 100
text = "NASA confirmed that the James Webb Space Telescope discovered signs of life on an exoplanet."

# PyTorch
t0 = time.perf_counter()
for _ in range(N):
    score_pytorch(text)
pt_ms = (time.perf_counter() - t0) / N * 1000

# ONNX
t0 = time.perf_counter()
for _ in range(N):
    score_onnx(text)
ox_ms = (time.perf_counter() - t0) / N * 1000

print(f"PyTorch:      {pt_ms:.1f} ms/inference")
print(f"ONNX Runtime: {ox_ms:.1f} ms/inference")
print(f"Speedup:      {pt_ms / ox_ms:.1f}x")

PyTorch:      165.0 ms/inference
ONNX Runtime: 41.1 ms/inference
Speedup:      4.0x


## 6. Save tokenizer for ONNX-only inference

In [7]:
print("Files needed for ONNX-only inference:")
needed = ["model.onnx", "tokenizer.json", "tokenizer_config.json", "config.json"]
for name in needed:
    p = MODEL_DIR / name
    size = p.stat().st_size / 1024 / 1024 if p.exists() else -1
    status = f"{size:.1f} MB" if size >= 0 else "MISSING"
    print(f"  {name:<30} {status}")

print(f"\nTotal ONNX inference deps:")
print(f"  onnxruntime   ~50 MB (pip)")
print(f"  tokenizers    ~5 MB (pip)")
print(f"  model.onnx    ~350 MB (file)")
print(f"  vs PyTorch:   ~2 GB (torch) + ~700 MB (safetensors)")

Files needed for ONNX-only inference:
  model.onnx                     704.4 MB
  tokenizer.json                 8.0 MB
  tokenizer_config.json          0.0 MB
  config.json                    0.0 MB

Total ONNX inference deps:
  onnxruntime   ~50 MB (pip)
  tokenizers    ~5 MB (pip)
  model.onnx    ~350 MB (file)
  vs PyTorch:   ~2 GB (torch) + ~700 MB (safetensors)


## 7. Download

In [ ]:
import shutil

onnx_dir = Path("onnx_export")
onnx_dir.mkdir(exist_ok=True)
for name in ["model.onnx", "tokenizer.json", "tokenizer_config.json", "config.json"]:
    src = MODEL_DIR / name
    if src.exists():
        shutil.copy2(src, onnx_dir / name)

shutil.make_archive("claim_detector_onnx", "zip", str(onnx_dir))

'/teamspace/studios/this_studio/claim_detector_onnx.zip'